# Study 877 — GDPNow Revisions — the teardown

The predictive regression with a Newey-West slope *t*, the top/bottom-decile conditional test, the execution-lag robustness, the two-era cut, the permutation placebo, the costed timer, and the 20-seed synthetic control.

In [1]:
R = {'start': '2011-08-30', 'end': '2026-06-26', 'n': 2042, 'n_forecasts': 2102, 'n_qtr': 60, 'fingerprint': '6161199ecd72', 'b1': -3.22, 't1': -1.02, 'r2_1': 0.043, 'b5': 1.47, 't5': 0.15, 'r2_5': 0.002, 'b1_l1': 3.27, 't1_l1': 0.89, 'b5_l1': -3.08, 't5_l1': -0.36, 'base_bps': 5.56, 'up_thr': 0.3, 'down_thr': -0.35, 'up_n': 205, 'up_bps': -18.71, 'up_t': -2.74, 'down_n': 205, 'down_bps': 0.09, 'down_t': 0.01, 'updown_welch': -1.64, 'up_l1_bps': 11.24, 'up_l1_t': 1.33, 'era_e_b': 1.99, 'era_e_t': 0.19, 'era_e_n': 918, 'era_l_b': -3.57, 'era_l_t': -1.08, 'era_l_n': 1124, 'placebo_obs': -3.22, 'placebo_sd': 3.5, 'placebo_p': 0.316, 'timer1_net': 0.36, 'timer1_ann': 0.9, 'timer1_sharpe': 0.07, 'timer5_net': -1.68, 'timer5_ann': -4.2, 'timer5_sharpe': -0.34, 'bh_sharpe': 0.8, 'exposure': 0.48, 'null_mean_t': -0.18, 'null_sd_t': 1.01, 'null_fire': 0, 'planted_t': 6.65, 'planted_b': 41.1, 'planted_r2': 2.14}

## The headline — forward SPY return on the revision (HAC t)

In [2]:
print(f"1-day fwd: beta {R['b1']:+.2f} bps/pp  NW t = {R['t1']:+.2f}  "
      f"R2 = {R['r2_1']:.3f}%  (n = {R['n']})")
print(f"5-day fwd: beta {R['b5']:+.2f} bps/pp  NW t = {R['t5']:+.2f}  "
      f"R2 = {R['r2_5']:.3f}%")
print(f"lag-1 robustness: 1-day beta {R['b1_l1']:+.2f} (t {R['t1_l1']:+.2f}) "
      f"-> SIGN FLIPS vs lag 0: no stable slope")

1-day fwd: beta -3.22 bps/pp  NW t = -1.02  R2 = 0.043%  (n = 2042)
5-day fwd: beta +1.47 bps/pp  NW t = +0.15  R2 = 0.002%
lag-1 robustness: 1-day beta +3.27 (t +0.89) -> SIGN FLIPS vs lag 0: no stable slope


## Decile conditional — biggest up vs biggest down revisions (fwd 1-day)

The claim: up-revisions → strength, down-revisions → weakness. The tape says the reverse-or-nothing.

In [3]:
print(f"base forward 1-day: {R['base_bps']:+.2f} bps")
print(f"top-decile UP  (rev>={R['up_thr']:+.3f}): n={R['up_n']}  "
      f"{R['up_bps']:+.2f} bps  NW t = {R['up_t']:+.2f}")
print(f"bot-decile DOWN(rev<={R['down_thr']:+.3f}): n={R['down_n']}  "
      f"{R['down_bps']:+.2f} bps  NW t = {R['down_t']:+.2f}")
print(f"up-minus-down Welch t = {R['updown_welch']:+.2f}")
print(f"[lag 1] top-decile UP flips to {R['up_l1_bps']:+.2f} bps "
      f"(t = {R['up_l1_t']:+.2f}) -> fragile intraday artefact")

base forward 1-day: +5.56 bps
top-decile UP  (rev>=+0.300): n=205  -18.71 bps  NW t = -2.74
bot-decile DOWN(rev<=-0.350): n=205  +0.09 bps  NW t = +0.01
up-minus-down Welch t = -1.64
[lag 1] top-decile UP flips to +11.24 bps (t = +1.33) -> fragile intraday artefact


## Robustness — two eras (split 2019-01-01)

In [4]:
print(f"2011-2018 (n={R['era_e_n']}): beta {R['era_e_b']:+.2f} bps  NW t = {R['era_e_t']:+.2f}")
print(f"2019-2026 (n={R['era_l_n']}): beta {R['era_l_b']:+.2f} bps  NW t = {R['era_l_t']:+.2f}")
print('the (already-insignificant) slope changes sign across halves')

2011-2018 (n=918): beta +1.99 bps  NW t = +0.19
2019-2026 (n=1124): beta -3.57 bps  NW t = -1.08
the (already-insignificant) slope changes sign across halves


## Placebo — shuffle forward returns against revisions (5,000 draws)

In [5]:
print(f"observed {R['placebo_obs']:+.2f} bps vs shuffled sd {R['placebo_sd']:.2f} "
      f"-> two-sided p = {R['placebo_p']:.3f}")

observed -3.22 bps vs shuffled sd 3.50 -> two-sided p = 0.316


## The timer — long SPY one day after an up-revision, flat otherwise

In [6]:
for tag,net,ann,shp in [('1 bp',R['timer1_net'],R['timer1_ann'],R['timer1_sharpe']),
                        ('5 bps',R['timer5_net'],R['timer5_ann'],R['timer5_sharpe'])]:
    print(f"{tag:>5} cost: net {net:+.2f} bps/day ({ann:+.1f}%/yr, Sharpe {shp:+.2f})")
print(f"vs buy-and-hold Sharpe {R['bh_sharpe']:.2f} on the same dates "
      f"(rule in the market only {R['exposure']*100:.0f}% of the time)")

 1 bp cost: net +0.36 bps/day (+0.9%/yr, Sharpe +0.07)
5 bps cost: net -1.68 bps/day (-4.2%/yr, Sharpe -0.34)
vs buy-and-hold Sharpe 0.80 on the same dates (rule in the market only 48% of the time)


## Synthetic positive control — the machinery is unbiased

Live: the regression must NOT fire on the null and must recover a planted revision→return edge.

In [7]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from gdpnow import data, strategy as st
null_t = np.array([st.synthetic_detect(data.synthetic(edge=0.0, seed=877+s, n=1500))['t'] for s in range(8)])
print(f"null (edge=0), 8 seeds: NW t mean {null_t.mean():+.2f} (sd {null_t.std(ddof=1):.2f}), |t|>=2 in {(abs(null_t)>=2).sum()}/8")
planted = st.synthetic_detect(data.synthetic(edge=0.005, seed=877, n=2000))
print(f"planted (edge=0.005): beta {planted['beta_bps']:+.1f} bps, NW t = {planted['t']:+.2f}, R2 = {planted['r2']*100:.2f}%")

null (edge=0), 8 seeds: NW t mean +0.26 (sd 1.14), |t|>=2 in 1/8
planted (edge=0.005): beta +41.1 bps, NW t = +6.65, R2 = 2.14%


## Verdict

- **Signal — None.** The GDPNow revision does **not** predict forward SPY with the claimed sign: 1-day slope NW *t* = **-1.02**, *R²* = **0.043%**, sign-flipping under a one-day lag and across eras (*t* = +0.19 / -1.08), inside a permutation placebo (*p* = 0.32). The only significant piece — top-decile **up**-revisions preceding next-day **weakness** (-18.71 bps, *t* = -2.74) — is **wrong-signed** vs the claim and flips to +11.24 bps once you can't trade the release-day close. Down-revisions are flat (*t* = +0.01). The 20-seed synthetic control fires on 0/20 nulls and recovers a planted edge (*t* = +6.65), so the flat real result is genuine, not a bug.
- **Tradability — Mirage.** An up-revision timer earns Sharpe **0.07** at 1 bp (vs 0.80 buy-and-hold) and goes negative (-0.34) at 5 bps.